## **IMDB Sentiment Analysis using NLP Pipeline & ML Models**

# **Import Libraries**

In [21]:
import pandas as pd
import numpy as np

# NLP
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# ML
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


# **Load data**

In [22]:
df = pd.read_csv("IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


# **Basic Checks**

In [5]:
print(df.shape)

(162980, 2)


In [6]:
print(df.columns)

Index(['clean_text', 'category'], dtype='object')


In [16]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [17]:
df.tail()

,review,sentiment
31049,I thought this movie did a down right good job...,positive
31050,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
31051,I am a Catholic taught in parochial elementary...,negative
31052,I'm going to have to disagree with the previou...,negative
31053,No one expects the Star Trek movies to be high...,negative


In [19]:
print(df.columns)

Index(['review', 'sentiment'], dtype='object')


In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31054 entries, 0 to 31053
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     31054 non-null  object
 1   sentiment  31054 non-null  object
dtypes: object(2)
memory usage: 485.3+ KB


In [24]:
print(df['sentiment'].value_counts())

sentiment
positive    15631
negative    15423
Name: count, dtype: int64


# **NLP Preprocessing**

In [25]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

# **Create preprocessing function**

In [46]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
    text = text.lower()  # Convert to lowercase

    text = re.sub(r"http\S+", "", text)  # Remove URLs
    text = re.sub(r"[^a-zA-Z]", " ", text)  # Remove special characters

    words = text.split()  # Tokenization (split sentence into words)

    words = [w for w in words if w not in stop_words]  # Remove stopwords

    words = [stemmer.stem(w) for w in words]  # Stemming

    return " ".join(words)

# **Apply preprocessing**

In [28]:
print(df.columns)


Index(['review', 'sentiment'], dtype='object')


In [29]:
df['clean_text'] = df['review'].apply(preprocess_text)

df[['review', 'clean_text']].head()

,review,clean_text
0,One of the other reviewers has mentioned that ...,one review mention watch oz episod hook right ...
1,A wonderful little production. <br /><br />The...,wonder littl product br br film techniqu unass...
2,I thought this was a wonderful way to spend ti...,thought wonder way spend time hot summer weeke...
3,Basically there's a family where a little boy ...,basic famili littl boy jake think zombi closet...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter mattei love time money visual stun film...


# **Feature Engineering**

In [40]:
bow = CountVectorizer(max_features=5000)

X_bow = bow.fit_transform(df['clean_text'])
X_bow.shape

(31054, 5000)

# **TF-IDF**

In [42]:
tfidf = TfidfVectorizer(max_features=5000)

X_tfidf = tfidf.fit_transform(df['clean_text'])
X_tfidf.shape

(31054, 5000)

# **Train-Test-Split**

In [60]:
#Split bow
y = df['sentiment']

X_train_bow, X_test_bow, y_train_bow, y_test_bow= train_test_split(X_bow, y, test_size=0.2, random_state=42
)

In [64]:
#Train Model
lr_bow = LogisticRegression(max_iter=2000)
lr_bow.fit(X_train_bow, y_train_bow)

LogisticRegression(max_iter=2000)

In [65]:
# Logistic Regression with BoW
#predict
y_pred_lr_bow = lr_bow.predict(X_test_bow)


In [66]:
#Evaluate
evaluate(y_test_bow, y_pred_lr_bow)

Accuracy: 0.862824021896635
Precision: 0.8628257333336838
Recall: 0.862824021896635
F1 Score: 0.862824519744712
----------------------------------------


# **Model Building**

**1)Logistic Regression**

In [71]:
y = df['sentiment']

x_train, x_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42
)

In [73]:
lr = LogisticRegression()
lr.fit(x_train, y_train)

y_pred_lr = lr.predict(x_test)

**2)Naive Bayes**

In [74]:
nb = MultinomialNB()
nb.fit(x_train, y_train)

y_pred_nb = nb.predict(x_test)

**Decision Tree**

In [75]:
dt = DecisionTreeClassifier()
dt.fit(x_train, y_train)

y_pred_dt = dt.predict(x_test)

# **Model Evaluation**

In [36]:
def evaluate(y_test, y_pred):
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred, average='weighted'))
    print("Recall:", recall_score(y_test, y_pred, average='weighted'))
    print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))
    print("-"*40)

 **Evaluate all models**

In [37]:
print("Logistic Regression")
evaluate(y_test, y_pred_lr)

print("Naive Bayes")
evaluate(y_test, y_pred_nb)

print("Decision Tree")
evaluate(y_test, y_pred_dt)

Logistic Regression
Accuracy: 0.8826275962002899
Precision: 0.883024162270039
Recall: 0.8826275962002899
F1 Score: 0.8826104771155394
----------------------------------------
Naive Bayes
Accuracy: 0.8426984382547094
Precision: 0.8428234392866316
Recall: 0.8426984382547094
F1 Score: 0.8426948498986481
----------------------------------------
Decision Tree
Accuracy: 0.7146997262920625
Precision: 0.7147333323254207
Recall: 0.7146997262920625
F1 Score: 0.7147020041739877
----------------------------------------


# **Comparision Table**

In [38]:
results = []

models = {
    "Logistic Regression": y_pred_lr,
    "Naive Bayes": y_pred_nb,
    "Decision Tree": y_pred_dt
}

for name, pred in models.items():
    results.append([
        name,
        accuracy_score(y_test, pred),
        precision_score(y_test, pred, average='weighted'),
        recall_score(y_test, pred, average='weighted'),
        f1_score(y_test, pred, average='weighted')
    ])

comparison_df = pd.DataFrame(results, columns=[
    "Model", "Accuracy", "Precision", "Recall", "F1 Score"
])

comparison_df

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.882628,0.883024,0.882628,0.882610
1,Naive Bayes,0.842698,0.842823,0.842698,0.842695
2,Decision Tree,0.714700,0.714733,0.714700,0.714702


# **Best Model Code**

In [44]:
best_model = comparison_df.sort_values(by="F1 Score", ascending=False).iloc[0]
print(best_model)

Model        Logistic Regression
Accuracy                0.882628
Precision               0.883024
Recall                  0.882628
F1 Score                 0.88261
Name: 0, dtype: object


# **Final Insight**

- Logistic Regression performed best due to its ability to handle sparse text data.
- TF-IDF gave better results than Bag of Words because it reduces importance of common words.
- Preprocessing (stopword removal + stemming) improved accuracy significantly.
- Naive Bayes is fast but slightly less accurate.
- Decision Tree overfitted the data.